In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import math
import torch.nn.functional as F
from tqdm import tqdm
import logging
from mpl_toolkits.axes_grid1 import make_axes_locatable # Import for precise colorbar placement
from skimage.metrics import structural_similarity as ssim # Import SSIM
from scipy.ndimage import uniform_filter, sobel # For local metric calculations
from skimage.transform import resize # Import resize for super-resolution

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# --- Configurable Variables ---
config = {
    'num_epochs': 200,
    'batch_size': 8,
    'learning_rate': 1e-4, # Decreased learning rate
    'noise_steps': 400,
    'time_embedding_dim': 256,
    'target_img_size': (400, 400), # High-resolution target image size
    'scale_factor': 2, # Super-resolution scale factor (e.g., 2 for 2x SR)
    'model_save_path': "seismic_plume_super_resolution_model.pth", # Updated model save path
    'data_file_path': "04p07ful_Realized_1.npy",
    'uncond_prob': 0.1, # Probability of training unconditionally for CFG
    'guidance_scale_for_plot': 0.0, # Guidance scale for plotting (can be changed for experiments)
    'scheduler_mode': 'min',
    'scheduler_factor': 0.5,
    'scheduler_patience': 4,
    'scheduler_verbose': True,
    'plot_results_every_epoch': 3, # Changed plot frequency to every 3 epochs
    'metrics_window_shape': (7, 7), # Window size for local metrics (Semblance, Variance, Eigen-Coherence)
    'aoi_coords': (100, 100, 350, 350), # (row_start, col_start, row_end, col_end) for Area of Interest
    'histogram_bins': 50, # Number of bins for histograms
    'tv_loss_weight': 0.01, # Weight for Total Variation (smoothness) loss
}

# Ensure reproducibility for all random operations
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Global variable for colormap
GLOBAL_CMAP = 'seismic' # Set to 'gray' or 'seismic' or any other matplotlib colormap

# --- Normalization Routine ---
def normalize_slice(slice_data, q=99.99):
    """
    Normalize seismic slice by clipping extreme values symmetrically
    around 0 based on the given quantile (default: 99.99%).
    Returns the normalized data and the clip_val used for normalization.
    """
    valid_data = slice_data[~np.isnan(slice_data)].flatten()

    if valid_data.size == 0:
        logging.warning('Empty or all-NaN slice encountered!!! Returning zeros.')
        return np.zeros_like(slice_data, dtype=np.float32), 0.0

    clip_val = np.percentile(np.abs(valid_data), q)

    if clip_val == 0:
        logging.warning('Slice with zero percentile (or all zero data) encountered!!! Returning zeros.')
        return np.zeros_like(slice_data, dtype=np.float32), 0.0

    clipped = np.clip(slice_data, -clip_val, clip_val)
    normalized = clipped / clip_val

    return normalized.astype(np.float32), clip_val

# --- 1. Seismic Data Loading and Augmentation for Super-Resolution ---
class SeismicDataset(Dataset):
    def __init__(self, file_path, num_variations_per_file=1000, target_img_size=(256, 256), scale_factor=2):
        self.clean_base_img = self._load_and_preprocess_data(file_path, target_img_size)
        self.num_samples = num_variations_per_file
        self.img_size = self.clean_base_img.shape # This is the HR size
        self.target_img_size = target_img_size # HR size
        self.scale_factor = scale_factor
        self.lr_img_size = (target_img_size[0] // scale_factor, target_img_size[1] // scale_factor)

    def _load_and_preprocess_data(self, file_path, target_img_size):
        """
        Loads .npy data, handles 3D/2D input, normalizes it to [-1, 1],
        and resizes/transposes it as necessary.
        Assumes the loaded data represents the 'clean' seismic image (HR target).
        """
        try:
            data = np.load(file_path)
        except FileNotFoundError:
            print(f"Error: File not found at {file_path}. Please ensure the .npy file exists.")
            raise

        if data.ndim == 3:
            print(f"Loaded 3D data with shape: {data.shape}. Taking the middle slice along the first dimension.")
            clean_img = data[data.shape[0] // 2, :, :]
        elif data.ndim == 2:
            print(f"Loaded 2D data with shape: {data.shape}.")
            clean_img = data
        else:
            raise ValueError(f"Unsupported data dimension: {data.ndim}. Expected 2D or 3D NumPy array.")

        clean_img = clean_img.astype(np.float32)

        if clean_img.shape != target_img_size:
            print(f"Resizing image from {clean_img.shape} to {target_img_size} (HR target)")
            clean_img = resize(clean_img, target_img_size, anti_aliasing=True, preserve_range=True)

        print(f"Transposing image from {clean_img.shape} to {clean_img.T.shape}")
        clean_img = clean_img.T

        normalized_img, _ = normalize_slice(clean_img)
        return normalized_img

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # HR image is the clean base image
        hr_image = torch.from_numpy(self.clean_base_img[np.newaxis, :, :]) # (1, H_HR, W_HR)

        # Create LR image by downsampling the HR image
        # Use bilinear interpolation for downsampling
        lr_image_np = resize(self.clean_base_img, self.lr_img_size, anti_aliasing=True, preserve_range=True)
        lr_image = torch.from_numpy(lr_image_np[np.newaxis, :, :]) # (1, H_LR, W_LR)

        return hr_image, lr_image # Return HR target and LR input
        
# --- 2. Diffusion Model Components ---

class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

# --- NEW: Squeeze-and-Excitation (SE) Block ---
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation block for channel-wise attention.
    It adaptively recalibrates channel-wise feature responses by explicitly modeling
    interdependencies between channels.
    """
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1) # Squeeze operation
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid() # Excitation operation
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c) # Squeeze: Global average pooling
        y = self.fc(y).view(b, c, 1, 1) # Excitation: Learn channel weights
        return x * y.expand_as(x) # Scale features with learned weights


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_embedding_dim, use_se=True):
        super().__init__()
        self.time_mlp = nn.Linear(time_embedding_dim, out_channels)

        # Changed kernel_size to 5 and padding to 2, added padding_mode='reflect'
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2, padding_mode='reflect')
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.LeakyReLU(0.01, inplace=True)
        # Changed kernel_size to 5 and padding to 2, added padding_mode='reflect'
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=5, padding=2, padding_mode='reflect')
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.use_se = use_se
        if self.use_se:
            self.se_block = SEBlock(out_channels)

        # Residual connection: 1x1 conv if dimensions change
        self.residual_conv = None
        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False)


    def forward(self, x, t_emb):
        identity = x # Store input for residual connection

        t_emb_reshaped = self.time_mlp(t_emb)[:, :, None, None]
        
        out = self.relu(self.bn1(self.conv1(x)))
        out = out + t_emb_reshaped # Add time conditioning
        out = self.relu(self.bn2(self.conv2(out)))

        if self.use_se:
            out = self.se_block(out)

        # Apply residual connection
        if self.residual_conv is not None:
            identity = self.residual_conv(identity)
        
        out = out + identity

        p = self.pool(out) # Pooling happens AFTER the block's operations and residual.
        return out, p # Return 'out' for skip connection, and 'p' for next down block


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_embedding_dim, use_se=True):
        super().__init__()
        self.time_mlp = nn.Linear(time_embedding_dim, out_channels)

        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        # Changed kernel_size to 5 and padding to 2, added padding_mode='reflect'
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2, padding_mode='reflect') # Input channels need to consider concatenation
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.LeakyReLU(0.01, inplace=True)
        # Changed kernel_size to 5 and padding to 2, added padding_mode='reflect'
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=5, padding=2, padding_mode='reflect')
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.use_se = use_se
        if self.use_se:
            self.se_block = SEBlock(out_channels)
        
        # Residual connection for the UpBlock's main convolutional path
        # The input to this residual is after concatenation and upsampling, before the convs
        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False) # 1x1 conv for residual


    def forward(self, x, skip_connection, t_emb):
        x = self.up(x) # x is now (batch, in_channels // 2, H_new, W_new)
        
        # Pad or crop skip_connection if dimensions don't match exactly
        diffY = skip_connection.size()[2] - x.size()[2]
        diffX = skip_connection.size()[3] - x.size()[3]
        x = nn.functional.pad(x, [diffX // 2, diffX - diffX // 2,
                                  diffY // 2, diffY - diffY // 2])

        combined_features = torch.cat([x, skip_connection], dim=1) # Combined is (batch, in_channels, H_new, W_new)
        
        identity = self.residual_conv(combined_features) # Residual path

        out = self.relu(self.bn1(self.conv1(combined_features)))
        t_emb_reshaped = self.time_mlp(t_emb)[:, :, None, None]
        out = out + t_emb_reshaped
        out = self.relu(self.bn2(self.conv2(out)))
        
        if self.use_se:
            out = self.se_block(out)
        
        return out + identity


class DiffusionUNet(nn.Module):
    # in_channels changed to 2 for super-resolution (noisy_hr_image + lr_condition)
    def __init__(self, in_channels=2, out_channels=1, features=[64, 128, 256], time_embedding_dim=256, use_se=True):
        super().__init__()
        self.time_embedding = SinusoidalPositionalEmbedding(time_embedding_dim)

        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()

        # Encoder part
        current_in_channels = in_channels
        for feature in features:
            self.downs.append(DownBlock(current_in_channels, feature, time_embedding_dim, use_se=use_se))
            current_in_channels = feature

        # Bottleneck - also gets SE and residual
        self.bottleneck = nn.Sequential(
            nn.Conv2d(features[-1], features[-1] * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(features[-1] * 2),
            nn.LeakyReLU(0.01, inplace=True),
            SEBlock(features[-1] * 2) if use_se else nn.Identity(), # Add SE block to bottleneck
            nn.Conv2d(features[-1] * 2, features[-1] * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(features[-1] * 2),
            nn.LeakyReLU(0.01, inplace=True),
            SEBlock(features[-1] * 2) if use_se else nn.Identity(), # Add SE block to bottleneck
        )

        # Decoder part
        self.ups.append(UpBlock(features[-1] * 2, features[-1], time_embedding_dim, use_se=use_se))
        
        for i in reversed(range(len(features) - 1)):
            self.ups.append(UpBlock(features[i+1], features[i], time_embedding_dim, use_se=use_se))

        # Final convolutional layer (removed Tanh activation here)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)

        skip_connections = []
        # Encoder Forward Pass
        for down in self.downs:
            s, x = down(x, t_emb) # s is output before pooling, x is output after pooling
            skip_connections.append(s)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1] # Reverse skip connections for decoder

        # Decoder Forward Pass
        for idx, up in enumerate(self.ups):
            x = up(x, skip_connections[idx], t_emb)

        return self.final_conv(x)


# --- 3. Diffusion Process Utilities (No changes needed here for architectural hacks) ---
class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=(256, 256), device="cpu"):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size # This is the HR image size
        self.device = device

        self.betas = self._prepare_noise_schedule().to(device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)

    def _prepare_noise_schedule(self):
        return torch.linspace(self.beta_start, self.beta_end, self.noise_steps)

    def noise_images(self, x_start, t):
        """
        Forward diffusion process: adds noise to x_start (HR clean image) at timestep t.
        x_start: clean HR image (batch_size, 1, H_HR, W_HR)
        t: timestep (batch_size,)
        """
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        
        epsilon = torch.randn_like(x_start)
        
        x_noisy = sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * epsilon
        
        return x_noisy, epsilon

    @torch.no_grad()
    def sample(self, model, n_samples, img_size, channels=1, lr_condition=None, guidance_scale=3.0):
        """
        Reverse diffusion process: samples HR images conditioned on LR input.
        Starts from pure random noise for HR and iteratively denoises it.
        lr_condition: The low-resolution image tensor (batch_size, 1, H_LR, W_LR) to condition the generation.
        guidance_scale (float): The scale for Classifier-Free Guidance. Higher values mean stronger guidance.
        """
        model.eval()
        if lr_condition is None:
            raise ValueError("lr_condition must be provided for super-resolution sampling.")

        # Ensure lr_condition is on the correct device and has the correct batch size
        lr_condition = lr_condition.to(self.device)
        n_samples = lr_condition.shape[0]

        print(f"Sampling {n_samples} HR images of size {img_size[0]}x{img_size[1]} conditioned on LR input...")
        # Start with pure noise for the HR image
        x = torch.randn((n_samples, channels, img_size[0], img_size[1]), device=self.device)

        # Upsample lr_condition to match the HR image size for concatenation
        # Use bilinear interpolation for upsampling
        upsampled_lr_condition = F.interpolate(lr_condition, size=img_size, mode='bilinear', align_corners=False)

        for i in tqdm(reversed(range(1, self.noise_steps)), desc="Sampling", total=self.noise_steps - 1):
            t_cond = torch.full((n_samples,), i, device=self.device, dtype=torch.long)
            t_uncond = torch.zeros_like(t_cond) # Timestep 0 for unconditional guidance

            # Conditional prediction: input is noisy HR + upsampled LR condition
            model_input_cond = torch.cat([x, upsampled_lr_condition], dim=1)
            predicted_noise_cond = model(model_input_cond, t_cond)
            
            # Unconditional prediction: input is noisy HR + zeroed upsampled LR condition
            # For unconditional path, the LR condition should be a tensor of zeros
            zero_upsampled_lr_condition = torch.zeros_like(upsampled_lr_condition)
            model_input_uncond = torch.cat([x, zero_upsampled_lr_condition], dim=1)
            predicted_noise_uncond = model(model_input_uncond, t_uncond)

            # Apply Classifier-Free Guidance
            predicted_noise = predicted_noise_uncond + guidance_scale * (predicted_noise_cond - predicted_noise_uncond)

            alpha = self.alphas[t_cond][:, None, None, None] # Use t_cond for alpha lookup
            alpha_cumprod = self.alphas_cumprod[t_cond][:, None, None, None]
            beta = self.betas[t_cond][:, None, None, None]
            posterior_variance_t = self.posterior_variance[t_cond][:, None, None, None]

            denoised_image = (x - predicted_noise * ((1 - alpha) / torch.sqrt(1 - alpha_cumprod))) / torch.sqrt(alpha)

            if i > 1:
                z = torch.randn_like(denoised_image)
                denoised_image = denoised_image + torch.sqrt(posterior_variance_t) * z
            
            x = denoised_image

        model.train()
        # Re-added torch.clamp here as Tanh activation is removed from final_conv
        return torch.clamp(x, -1, 1)

# --- New: Seismic Attribute Calculation Functions ---
def calculate_local_metrics(image_np, window_shape=(7, 7)):
    """
    Calculates mean semblance, mean local variance, and mean eigenstructure-based coherence
    for a 2D image using a sliding window.
    
    Args:
        image_np (np.ndarray): The 2D input image (NumPy array).
        window_shape (tuple): The (height, width) of the sliding window.

    Returns:
        tuple: (mean_semblance, mean_variance, mean_eigen_coherence)
    """
    H, W = image_np.shape
    
    # Pad image to handle borders for windowing
    pad_h = window_shape[0] // 2
    pad_w = window_shape[1] // 2
    padded_image = np.pad(image_np, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')

    # --- Semblance (simplified for 2D image) ---
    # Sum of amplitudes in window
    sum_amp = uniform_filter(padded_image, size=window_shape, mode='constant', cval=0.0)
    # Sum of squared amplitudes in window
    sum_sq_amp = uniform_filter(padded_image**2, size=window_shape, mode='constant', cval=0.0)

    # Crop back to original size
    sum_amp = sum_amp[pad_h:H+pad_h, pad_w:W+pad_w]
    sum_sq_amp = sum_sq_amp[pad_h:H+pad_h, pad_w:W+pad_w]

    N = window_shape[0] * window_shape[1]
    
    semblance_numerator = sum_amp**2
    semblance_denominator = N * sum_sq_amp
    
    semblance = np.where(semblance_denominator != 0, semblance_numerator / semblance_denominator, 0.0)
    semblance = np.clip(semblance, 0.0, 1.0) # Ensure values are within [0, 1]

    # --- Variance ---
    # Local mean
    local_mean = uniform_filter(padded_image, size=window_shape, mode='constant', cval=0.0)
    # Local mean of squares
    local_mean_sq = uniform_filter(padded_image**2, size=window_shape, mode='constant', cval=0.0)
    
    # Crop back to original size
    local_mean = local_mean[pad_h:H+pad_h, pad_w:W+pad_w]
    local_mean_sq = local_mean_sq[pad_h:H+pad_h, pad_w:W+pad_w]

    local_variance = local_mean_sq - local_mean**2
    local_variance = np.clip(local_variance, 0.0, None) # Variance cannot be negative

    # --- Eigenstructure-based Coherence (simplified structure tensor approach) ---
    # Compute gradients using Sobel filter
    dx = sobel(padded_image, axis=1, mode='constant', cval=0.0)
    dy = sobel(padded_image, axis=0, mode='constant', cval=0.0)

    # Structure tensor components
    Jxx = dx * dx
    Jyy = dy * dy
    Jxy = dx * dy

    # Smooth structure tensor components over the window
    Jxx_smooth = uniform_filter(Jxx, size=window_shape, mode='constant', cval=0.0)
    Jyy_smooth = uniform_filter(Jyy, size=window_shape, mode='constant', cval=0.0)
    Jxy_smooth = uniform_filter(Jxy, size=window_shape, mode='constant', cval=0.0)

    # Crop back to original size
    Jxx_smooth = Jxx_smooth[pad_h:H+pad_h, pad_w:W+pad_w]
    Jyy_smooth = Jyy_smooth[pad_h:H+pad_h, pad_w:W+pad_w]
    Jxy_smooth = Jxy_smooth[pad_h:H+pad_h, pad_w:W+pad_w]

    # Calculate eigenvalues for each pixel
    discriminant = (Jxx_smooth - Jyy_smooth)**2 + 4 * Jxy_smooth**2
    discriminant = np.maximum(discriminant, 0) # Ensure non-negative
    
    sqrt_discriminant = np.sqrt(discriminant)
    
    lambda1 = 0.5 * (Jxx_smooth + Jyy_smooth + sqrt_discriminant)
    lambda2 = 0.5 * (Jxx_smooth + Jyy_smooth - sqrt_discriminant)

    # Coherence definition: (lambda1 - lambda2) / (lambda1 + lambda2 + epsilon)
    epsilon = 1e-8
    eigen_coherence = (lambda1 - lambda2) / (lambda1 + lambda2 + epsilon)
    eigen_coherence = np.clip(eigen_coherence, 0.0, 1.0) # Coherence should be between 0 and 1

    # Return mean values of the attribute maps
    return np.mean(semblance), np.mean(local_variance), np.mean(eigen_coherence)


# --- 4. Plotting Routine ---
def plot_results(epoch, original_hr_img, lr_input_img, generated_sr_img, psnr_val, ssim_val, rms_contrast_orig, rms_contrast_gen, maa_orig, maa_gen, mgm_orig, mgm_gen, semb_orig, semb_gen, var_orig, var_gen, eigcoh_orig, eigcoh_gen, aoi_coords, save_dir="results"):
    """
    Plots the original HR input, the LR input, the diffusion model's generated SR output, and the residue
    side-by-side. Includes a second row for AOI.
    Saves the plot as a PNG image and displays it inline.
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    original_hr_img_np = original_hr_img.squeeze().cpu().numpy()
    lr_input_img_np = lr_input_img.squeeze().cpu().numpy()
    generated_sr_img_np = generated_sr_img.squeeze().cpu().numpy()
    
    # Calculate residue
    residue_img_np = original_hr_img_np - generated_sr_img_np 

    # Extract AOI patches
    r_start, c_start, r_end, c_end = aoi_coords
    original_aoi_np = original_hr_img_np[r_start:r_end, c_start:c_end]
    # For LR AOI, we need to adjust coordinates based on scale factor
    scale_factor = original_hr_img_np.shape[0] // lr_input_img_np.shape[0]
    lr_r_start, lr_c_start = r_start // scale_factor, c_start // scale_factor
    lr_r_end, lr_c_end = r_end // scale_factor, c_end // scale_factor
    lr_aoi_np = lr_input_img_np[lr_r_start:lr_r_end, lr_c_start:lr_c_end]

    generated_aoi_np = generated_sr_img_np[r_start:r_end, c_start:c_end]
    residue_aoi_np = residue_img_np[r_start:r_end, c_start:c_end]


    # Adjust figsize for two rows of images (now 4 columns)
    fig, axes = plt.subplots(2, 4, figsize=(24, 12)) # Two rows, four columns

    # Define a common vmin and vmax for all seismic images
    common_vmin = -1.0
    common_vmax = 1.0

    # Row 1: Full Images
    # Original HR Input
    im0 = axes[0, 0].imshow(original_hr_img_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[0, 0].set_title(f'Epoch {epoch} - Original HR Input\nRMS Contrast: {rms_contrast_orig:.4f}')

    # LR Input
    im_lr = axes[0, 1].imshow(lr_input_img_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[0, 1].set_title(f'Epoch {epoch} - LR Input ({lr_input_img_np.shape[0]}x{lr_input_img_np.shape[1]})')

    # Generated SR Output
    im1 = axes[0, 2].imshow(generated_sr_img_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[0, 2].set_title(f'Epoch {epoch} - Generated SR Output\nRMS Contrast: {rms_contrast_gen:.4f}')

    # Residue
    im2 = axes[0, 3].imshow(residue_img_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[0, 3].set_title(f'Epoch {epoch} - Residue (HR - SR)\nRMS Contrast Diff: {rms_contrast_orig - rms_contrast_gen:.4f}')
    
    # Add a single common color bar to the right of the last plot (residue)
    divider = make_axes_locatable(axes[0, 3])
    cax = divider.append_axes("right", size="2%", pad=0.1) # size="2%" makes it thinner
    cbar = fig.colorbar(im2, cax=cax)
    cbar.set_label('Amplitude')
    cbar.set_ticks([common_vmin, 0, common_vmax]) # Ensure colorbar reflects the common scale

    # Row 2: AOI Plots
    # Original HR AOI
    im_aoi0 = axes[1, 0].imshow(original_aoi_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[1, 0].set_title(f'AOI: Original HR ({r_start}:{r_end}, {c_start}:{c_end})')

    # LR AOI
    im_aoi_lr = axes[1, 1].imshow(lr_aoi_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[1, 1].set_title(f'AOI: LR Input ({lr_r_start}:{lr_r_end}, {lr_c_start}:{lr_c_end})')

    # Generated SR AOI
    im_aoi1 = axes[1, 2].imshow(generated_aoi_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[1, 2].set_title(f'AOI: Generated SR ({r_start}:{r_end}, {c_start}:{c_end})')

    # Residue AOI
    im_aoi2 = axes[1, 3].imshow(residue_aoi_np, cmap=GLOBAL_CMAP, vmin=common_vmin, vmax=common_vmax)
    axes[1, 3].set_title(f'AOI: Residue ({r_start}:{r_end}, {c_start}:{c_end})')

    # Add a single common color bar for AOI plots
    divider_aoi = make_axes_locatable(axes[1, 3])
    cax_aoi = divider_aoi.append_axes("right", size="2%", pad=0.1)
    cbar_aoi = fig.colorbar(im_aoi2, cax=cax_aoi)
    cbar_aoi.set_label('Amplitude')
    cbar_aoi.set_ticks([common_vmin, 0, common_vmax])


    # Add metrics summary at the bottom of the plot
    metrics_text = (
        f"PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}\n"
        f"Orig HR MAA: {maa_orig:.4f} | Gen SR MAA: {maa_gen:.4f} | "
        f"Orig HR MGM: {mgm_orig:.4f} | Gen SR MGM: {mgm_gen:.4f}\n"
        f"Orig HR Semblance: {semb_orig:.4f} | Gen SR Semblance: {semb_gen:.4f}\n"
        f"Orig HR Variance: {var_orig:.4f} | Gen SR Variance: {var_gen:.4f}\n"
        f"Orig HR Eigen-Coherence: {eigcoh_orig:.4f} | Gen SR Eigen-Coherence: {eigcoh_gen:.4f}"
    )
    # Adjust position to be at the bottom, further down
    # Increased the y-coordinate for figtext to move it further down
    plt.figtext(0.5, 0.02, metrics_text, ha='center', fontsize=10, bbox={"facecolor":"white", "alpha":0.8, "pad":5})

    plt.suptitle(f'Seismic Plume Super-Resolution with Diffusion - Epoch {epoch}', fontsize=16)
    # Adjust rect to make more space for bottom text and two rows of plots.
    plt.tight_layout(rect=[0, 0.1, 1, 0.95]) # Adjusted bottom margin to 0.1
    plt.savefig(os.path.join(save_dir, f'super_resolution_diffusion_epoch_{epoch:03d}.png'))
    plt.show()
    plt.close(fig)

# --- 5. Training Configuration and Loop ---

def train_diffusion_model(resume_training=False, target_shape=None):
    # Use target_shape from config if not provided in function call
    if target_shape is None:
        target_shape = config['target_img_size']

    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        torch.cuda.set_device(0)
        print(f"Using GPU device: {device}")
    else:
        device = torch.device("cpu")
        print(f"Using CPU device: {device}")

    num_epochs = config['num_epochs']
    batch_size = config['batch_size']
    learning_rate = config['learning_rate']
    noise_steps = config['noise_steps']
    time_embedding_dim = config['time_embedding_dim']
    model_save_path = config['model_save_path']
    data_file_path = config['data_file_path']
    uncond_prob = config['uncond_prob']
    guidance_scale_for_plot = config['guidance_scale_for_plot']
    aoi_coords = config['aoi_coords']
    histogram_bins = config['histogram_bins']
    tv_loss_weight = config['tv_loss_weight']
    scale_factor = config['scale_factor']


    dataset = SeismicDataset(file_path=data_file_path, num_variations_per_file=1000, target_img_size=target_shape, scale_factor=scale_factor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # img_size is the HR size
    sample_hr_img, sample_lr_img = dataset[0] 
    img_size = sample_hr_img.shape[1:] # HR image size
    # in_channels for DiffusionUNet is 2 (noisy HR + LR condition)
    channels_in_model = 2 
    channels_out_model = 1 # Output is still noise for HR image

    # Reverted features to original capacity
    model = DiffusionUNet(in_channels=channels_in_model, out_channels=channels_out_model, features=[64, 128, 256, 512, 1024], time_embedding_dim=time_embedding_dim, use_se=True).to(device)
    # Diffusion process operates on the HR image dimensions
    diffusion = Diffusion(noise_steps=noise_steps, beta_start=1e-4, beta_end=0.02, img_size=img_size, device=device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Learning Rate Scheduler
    scheduler = ReduceLROnPlateau(optimizer, mode=config['scheduler_mode'], factor=config['scheduler_factor'], patience=config['scheduler_patience'], verbose=config['scheduler_verbose'])

    start_epoch = 1
    if resume_training and os.path.exists(model_save_path):
        print(f"Loading previous model weights from {model_save_path}")
        checkpoint = torch.load(model_save_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if not hasattr(scheduler, 'last_epoch'):
                 scheduler.last_epoch = checkpoint['epoch']
            elif scheduler.last_epoch != checkpoint['epoch']:
                 scheduler.last_epoch = checkpoint['epoch']
        print(f"Resuming training from epoch {start_epoch}")
    elif resume_training and not os.path.exists(model_save_path):
        print(f"Resume training requested, but no model found at {model_save_path}. Starting from scratch.")
    else:
        print("Starting training from scratch (resume_training is False).")

    # Cache the sampled slice for consistent plotting
    print("Caching sampled slices for plotting...")
    if len(dataset) > 100:
        original_hr_for_plot_cached, lr_input_for_plot_cached = dataset[100]
    else:
        logging.warning("Dataset has less than 101 samples. Using dataset[0] for plotting.")
        original_hr_for_plot_cached, lr_input_for_plot_cached = dataset[0]
    
    original_hr_batch_cached = original_hr_for_plot_cached.unsqueeze(0).to(device)
    lr_input_batch_cached = lr_input_for_plot_cached.unsqueeze(0).to(device)
    print("Sampled slices cached.")


    print("Starting Diffusion Model training...")
    for epoch in range(start_epoch, num_epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{num_epochs}")
        for batch_idx, (clean_hr_images, lr_images) in enumerate(pbar):
            clean_hr_images = clean_hr_images.to(device)
            lr_images = lr_images.to(device) # LR images are the condition

            t = torch.randint(0, diffusion.noise_steps, (clean_hr_images.shape[0],), device=device).long()

            # Implement unconditional training: set t=0 for a fraction of samples
            uncond_mask = torch.rand(clean_hr_images.shape[0], device=device) < uncond_prob
            t[uncond_mask] = 0 # Use timestep 0 for unconditional samples

            # Add noise to the HR images
            noisy_hr_images, true_noise_for_hr_image = diffusion.noise_images(clean_hr_images, t)

            # Upsample lr_images to match the HR image size for concatenation
            upsampled_lr_images = F.interpolate(lr_images, size=img_size, mode='bilinear', align_corners=False)

            # Concatenate noisy HR image with upsampled LR image for model input
            # The model learns to predict noise for HR, conditioned on LR
            model_input = torch.cat([noisy_hr_images, upsampled_lr_images], dim=1)
            predicted_noise = model(model_input, t)

            # Primary MSE loss between predicted noise and true noise for HR image
            mse_loss = criterion(predicted_noise, true_noise_for_hr_image)

            # Calculate Total Variation (TV) Loss for smoothness
            tv_loss = torch.mean(torch.abs(predicted_noise[:, :, :, 1:] - predicted_noise[:, :, :, :-1])) + \
                      torch.mean(torch.abs(predicted_noise[:, :, 1:, :] - predicted_noise[:, :, :-1, :]))
            
            # Combine losses
            loss = mse_loss + tv_loss_weight * tv_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * clean_hr_images.size(0)

            pbar.set_postfix(loss=loss.item())

        epoch_loss = running_loss / len(dataset)
        # Print current learning rate
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch}/{num_epochs}], Loss: {epoch_loss:.6f}, Current LR: {current_lr:.6e}")

        scheduler.step(epoch_loss)

        if epoch % config['plot_results_every_epoch'] == 0:
            model.eval()
            with torch.no_grad():
                # Upsample the cached LR input for plotting before passing to sample
                upsampled_lr_input_for_plot_cached = F.interpolate(lr_input_batch_cached, size=img_size, mode='bilinear', align_corners=False)
                
                # Pass upsampled_lr_input_for_plot_cached to sample method for conditioning
                generated_sr_img = diffusion.sample(model, n_samples=1, img_size=img_size, channels=1, lr_condition=upsampled_lr_input_for_plot_cached, guidance_scale=guidance_scale_for_plot)

                # Convert to numpy for metric calculation
                original_hr_img_np = original_hr_batch_cached.squeeze().cpu().numpy()
                generated_sr_img_np = generated_sr_img.squeeze().cpu().numpy()
                lr_input_img_np = lr_input_batch_cached.squeeze().cpu().numpy() # For plotting LR input (original LR size)
                
                # Calculate residue here to ensure it's in scope for histograms
                residue_img_np = original_hr_img_np - generated_sr_img_np

                # Calculate PSNR
                mse_sampled = F.mse_loss(generated_sr_img, original_hr_batch_cached)
                MAX_I = 2.0
                psnr_sampled = 10 * torch.log10(MAX_I**2 / mse_sampled).item() if mse_sampled > 1e-8 else float('inf')
                
                # Calculate SSIM
                # data_range is max_pixel_value - min_pixel_value, which is 1 - (-1) = 2.0
                ssim_sampled = ssim(original_hr_img_np, generated_sr_img_np, data_range=2.0)

                # Calculate RMS Contrast (standard deviation of pixel values)
                rms_contrast_original = np.std(original_hr_img_np)
                rms_contrast_generated = np.std(generated_sr_img_np)

                # Calculate Mean Absolute Amplitude (MAA)
                maa_original = np.mean(np.abs(original_hr_img_np))
                maa_generated = np.mean(np.abs(generated_sr_img_np))

                # Calculate Mean Gradient Magnitude (MGM)
                mgm_original = np.mean(np.sqrt(np.gradient(original_hr_img_np)[0]**2 + np.gradient(original_hr_img_np)[1]**2))
                mgm_generated = np.mean(np.sqrt(np.gradient(generated_sr_img_np)[0]**2 + np.gradient(generated_sr_img_np)[1]**2))

                # Calculate Semblance, Variance, Eigenstructure-based Coherence
                semb_original, var_original, eigcoh_original = calculate_local_metrics(original_hr_img_np, window_shape=config['metrics_window_shape'])
                semb_generated, var_generated, eigcoh_generated = calculate_local_metrics(generated_sr_img_np, window_shape=config['metrics_window_shape'])


                print(f"Epoch {epoch} - PSNR: {psnr_sampled:.2f} dB | SSIM: {ssim_sampled:.4f} | Orig HR RMS Contrast: {rms_contrast_original:.4f} | Gen SR RMS Contrast: {rms_contrast_generated:.4f} | Orig HR MAA: {maa_original:.4f} | Gen SR MAA: {maa_generated:.4f} | Orig HR MGM: {mgm_original:.4f} | Gen SR MGM: {mgm_generated:.4f} | Orig HR Semblance: {semb_original:.4f} | Gen SR Semblance: {semb_generated:.4f} | Orig HR Variance: {var_original:.4f} | Gen SR Variance: {var_generated:.4f} | Orig HR Eigen-Coherence: {eigcoh_original:.4f} | Gen SR Eigen-Coherence: {eigcoh_generated:.4f}")

                # Pass LR input image (original LR size) to plot_results
                plot_results(epoch, original_hr_batch_cached, lr_input_batch_cached, generated_sr_img, psnr_sampled, ssim_sampled, rms_contrast_original, rms_contrast_generated, maa_original, maa_generated, mgm_original, mgm_generated, semb_original, semb_generated, var_original, var_generated, eigcoh_original, eigcoh_generated, aoi_coords)
            model.train()

            # --- Print Histograms ---
            bins = np.linspace(-1, 1, histogram_bins) # Use configurable bins
            
            hist_orig, bins_orig = np.histogram(original_hr_img_np.flatten(), bins=bins)
            hist_gen, bins_gen = np.histogram(generated_sr_img_np.flatten(), bins=bins)
            hist_residue, bins_residue = np.histogram(residue_img_np.flatten(), bins=bins)

            print(f"\n--- Histograms for Epoch {epoch} ---")
            print("Original HR Image Histogram (Counts, Bins):")
            print(hist_orig)
            print(bins_orig)
            print("\nGenerated SR Image Histogram (Counts, Bins):")
            print(hist_gen)
            print(bins_gen)
            print("\nResidue Image Histogram (Counts, Bins):")
            print(hist_residue)
            print(bins_residue)
            print("-------------------------------------\n")


        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': epoch_loss,
        }, model_save_path)
        print(f"Model checkpoint saved to {model_save_path} for epoch {epoch}")


    print("Diffusion Model training complete!")
    torch.save({
        'epoch': num_epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': epoch_loss,
    }, model_save_path)
    print("Final diffusion model saved.")


if __name__ == '__main__':
    train_diffusion_model(resume_training=False) # target_shape now comes from config by default
